# 02b · scGPT (whole-human) embeddings

**Environment:** `envs/scgpt.yml` (this foundation model has heavy, specific dependencies: run this notebook in its own environment).
**Output:** one `.h5ad` per dataset whose `X` is the frozen embedding (512-d) and whose `obs` carries `cell_id`, `cell_type`, `master_split`.
The backbone is frozen: no fine-tuning, no gradients.

Get the model: clone `https://github.com/bowang-lab/scGPT` and download the whole-human checkpoint into `pretrained model/` (needs `vocab.json`, `args.json`, `best_model.pt`).

> These extraction functions are ported from the authors' working notebooks and need the model's own code and checkpoint.
> They are not run in the repository's tests. Try them on the smallest dataset first and check the shapes at the end.

In [ ]:
from pathlib import Path
from heterosc.extraction.scgpt import extract_scgpt_cells

DATA = Path("data")                       # from notebook 01
OUT = Path("outputs/embeddings")          # shared layout: {dataset}/{expert}/{dataset}_{expert}_with_embeddings.h5ad
DATASETS = ["Myeloid"]                    # start small; then add Lung, Liver, Pancreas, Skin
SCGPT_REPO = Path("checkpoints/scGPT")

## Part A · annotation datasets

In [ ]:
for ds in DATASETS:
    extract_scgpt_cells(
        DATA / "annotation" / "processed" / f"{ds}_ready_master_split.h5ad",
        OUT / ds / "scgpt" / f"{ds}_scgpt_with_embeddings.h5ad",
        scgpt_repo=SCGPT_REPO)

## Part B · cell lines for drug response
Uses `data/drug_response/cellline_expression.h5ad` (CCLE expression, 697 genes). Output layout used by notebook 04:
`outputs/drug_response_embeddings/{expert}/cellline_{expert}_embeddings.h5ad`.

In [ ]:
extract_scgpt_cells(DATA / "drug_response" / "cellline_expression.h5ad",
                    "outputs/drug_response_embeddings/scgpt/cellline_scgpt_embeddings.h5ad",
                    scgpt_repo=SCGPT_REPO, extra_obs=())

## Check

In [ ]:
from heterosc.extraction import load_embeddings
import glob
for f in sorted(glob.glob("outputs/**/*_scgpt_*embeddings.h5ad", recursive=True)):
    X, obs = load_embeddings(f)
    print(f, X.shape, "(expected dim 512)")